# Laboratorio # 5

Vamos a leer el dataset

In [8]:
import pandas as pd


train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")


In [9]:
train.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [10]:
test.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


Vamos a empezar a limpiar el dataset. Primero vamos a convertir todo el texto a minúscula.

In [14]:
import re

def convert_to_lower(word):
    return word.lower()

def remove_special_characters(word):
    return word.replace("#", "").replace("@", "").replace("'", "")

def remove_url(word):
    text = re.sub(r'http\S+|www\S+|https\S+', '', word)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def remove_emojis(word):
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticonos 🙂
        "\U0001F300-\U0001F5FF"  # símbolos y pictogramas 🗻
        "\U0001F680-\U0001F6FF"  # transporte y mapas 🚗
        "\U0001F1E0-\U0001F1FF"  # banderas 🇬🇹
        "\U00002700-\U000027BF"  # símbolos varios ➡
        "\U0001F900-\U0001F9FF"  # suplementarios 🤖
        "\U00002600-\U000026FF"  # misceláneos ☀
        "\U00002B00-\U00002BFF"  # flechas ⬆
        "\U0001FA70-\U0001FAFF"  # emojis recientes 🪐
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub(r'', word).strip()

train['text'] = train['text'].astype(str).apply(convert_to_lower)
train['text'] = train['text'].astype(str).apply(remove_special_characters)
train['text'] = train['text'].astype(str).apply(remove_url)
train['text'] = train['text'].astype(str).apply(remove_emojis)


train.head()

,id,keyword,location,text,target
0,1,NaN,NaN,our deeds are the reason of this earthquake ma...,1
1,4,NaN,NaN,forest fire near la ronge sask. canada,1
2,5,NaN,NaN,all residents asked to shelter in place are be...,1
3,6,NaN,NaN,"13,000 people receive wildfires evacuation ord...",1
4,7,NaN,NaN,just got sent this photo from ruby alaska as s...,1


## Limpieza
### Quitar los signos de puntuacion


In [23]:
import re

train["text"] = train["text"].str.replace(r'[^\w\s]', '', regex=True)

test["text"] = test["text"].str.replace(r'[^\w\s]', '', regex=True)


### Quitar los artículos, preposiciones y conjunciones


In [24]:
import pandas as pd
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    tokens = text.split()  # separar en palabras
    tokens = [word for word in tokens if word.lower() not in stop_words]
    return " ".join(tokens)

# Aplicar a las columnas
train["text"] = train["text"].apply(remove_stopwords)
test["text"] = test["text"].apply(remove_stopwords)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mathew\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Quitar numeros

Primero vamos a ver cuales eliminar, listando por los que mas aparecen


In [25]:
import re
import pandas as pd

def extract_numbers(text):
    return re.findall(r'\d+', str(text))
train_nums = train.copy()
train_nums["numbers"] = train_nums["text"].apply(extract_numbers)

test_nums = test.copy()
test_nums["numbers"] = test_nums["text"].apply(extract_numbers)

all_nums = pd.concat([train_nums[["numbers"]], test_nums[["numbers"]]], ignore_index=True)

all_nums = all_nums[all_nums["numbers"].map(len) > 0]

import numpy as np
all_numbers_flat = np.concatenate(all_nums["numbers"].values)

summary = pd.Series(all_numbers_flat).value_counts().reset_index()
summary.columns = ["number", "count"]
summary


,number,count
0,1,960
1,2,957
2,3,943
3,5,933
4,4,926
...,...,...
591,720,1
592,964,1
593,518,1
594,72254,1


Sabiendo esto vamos a conservar los numeros 1945, 911, 2008, 2014, 1980, 2013, 2016, 2011  ya que son fechas importantes, ademas de que son numeros de telefono que pueden tener relacion con el analisis de sentimientos


In [26]:
import re

allowed_numbers = {"1945", "911", "2008", "2014", "1980", "2013", "2016", "2011"}

def remove_unwanted_numbers(text):
    return re.sub(r'\b(?!' + '|'.join(allowed_numbers) + r')\d+\b', '', str(text))

train["text"] = train["text"].apply(remove_unwanted_numbers)
test["text"] = test["text"].apply(remove_unwanted_numbers)


test.head()



,id,keyword,location,text,target
0,1,NaN,NaN,Deeds Reason earthquake May ALLAH Forgive us,1
1,4,NaN,NaN,Forest fire near La Ronge Sask Canada,1
2,5,NaN,NaN,residents asked shelter place notified officer...,1
3,6,NaN,NaN,people receive wildfires evacuation orders Cal...,1
4,7,NaN,NaN,got sent photo Ruby Alaska smoke wildfires pou...,1
